# NOTE: Remove this step in the future and replace with code in model stream

In [35]:
import pandas as pd
import os
import numpy as np

# import global TDM functions
import sys
sys.path.insert(0, '../Resources/2-Python/global-functions')
import BigQuery

client = BigQuery.getBigQueryClient_Confidential2023UtahHTS()

bhereth
C:\Users\bhereth\confidential-2023-utah-hts-db5335615978.json


# Get Home-Based Job percents from external process

In [36]:
wah_df = client.query("SELECT * FROM `wfrc-modeling-data.ext_rsg_wah_2023.control_total_work_at_home_2023`").to_dataframe()
wah_df

,_Index,CO_FIPS,YEAR,Tel_RETL,Tel_FOOD,Tel_MANU,Tel_WSLE,Tel_OFFI,Tel_GVED,Tel_HLTH,...,HBJ_GVED,HBJ_HLTH,HBJ_OTHR,HBJ_AGRI,HBJ_MING,HBJ_CONS,HBJ,Telecom,Tot_WAH,CO_NAME
0,392023,39,2023,0.062,0.031,0.024,0.074,0.177,0.045,0.041,...,0.027,0.025,0.074,0.001,0.001,0.021,0.036,0.060,0.096,"Sanpete County, Utah"
1,252023,25,2023,0.038,0.043,0.162,0.136,0.217,0.206,0.271,...,0.015,0.020,0.015,0.005,0.005,0.023,0.012,0.159,0.170,"Kane County, Utah"
2,72023,7,2023,0.043,0.017,0.011,0.136,0.152,0.024,0.025,...,0.008,0.008,0.014,0.008,0.008,0.010,0.014,0.041,0.055,"Carbon County, Utah"
3,12023,1,2023,0.076,0.052,0.093,0.136,0.083,0.054,0.054,...,0.074,0.074,0.033,0.011,0.011,0.051,0.061,0.044,0.105,"Beaver County, Utah"
4,452023,45,2023,0.065,0.050,0.037,0.064,0.258,0.139,0.163,...,0.029,0.034,0.015,0.013,0.013,0.009,0.024,0.113,0.137,"Tooele County, Utah"
5,472023,47,2023,0.019,0.007,0.054,0.083,0.174,0.012,0.015,...,0.010,0.012,0.019,0.013,0.013,0.014,0.029,0.035,0.065,"Uintah County, Utah"
6,412023,41,2023,0.069,0.003,0.128,0.136,0.128,0.058,0.066,...,0.029,0.033,0.026,0.014,0.014,0.030,0.032,0.063,0.095,"Sevier County, Utah"
7,52023,5,2023,0.072,0.058,0.055,0.097,0.219,0.076,0.076,...,0.021,0.021,0.031,0.015,0.015,0.036,0.028,0.101,0.129,"Cache County, Utah"
8,352023,35,2023,0.106,0.061,0.101,0.155,0.279,0.126,0.121,...,0.024,0.023,0.028,0.016,0.016,0.016,0.028,0.149,0.177,"Salt Lake County, Utah"
9,292023,29,2023,0.083,0.033,0.023,0.136,0.269,0.044,0.039,...,0.015,0.013,0.058,0.021,0.021,0.024,0.031,0.091,0.122,"Morgan County, Utah"


In [37]:
# normalize WAH data to long format

# get list of HBJ columns
hbj_cols = [c for c in wah_df.columns if c.startswith("HBJ_")]

# filter to only necessary columns
wah_hbj_df = wah_df[['CO_FIPS', 'YEAR'] + hbj_cols]

# melt the HBJ columns to long format
wah_melted_df = (
    wah_hbj_df
      .melt(
          id_vars=[c for c in wah_hbj_df.columns if not c.startswith("HBJ_")],
          value_vars=hbj_cols,
          var_name="HBJFULL",
          value_name="HBJPCT"
      )
      .assign(
          TDM_SECTOR=lambda d: d["HBJFULL"].str.split("_").str[1]
      )
      .drop(columns=["HBJFULL"])
).sort_values(by=['CO_FIPS', 'YEAR', 'TDM_SECTOR']).reset_index(drop=True)

# select and reorder final columns
wah_melted_df = wah_melted_df[['CO_FIPS', 'YEAR', 'TDM_SECTOR', 'HBJPCT']]

wah_melted_df

,CO_FIPS,YEAR,TDM_SECTOR,HBJPCT
0,1,2023,AGRI,0.011
1,1,2023,CONS,0.051
2,1,2023,FOOD,0.012
3,1,2023,GVED,0.074
4,1,2023,HLTH,0.074
...,...,...,...,...
325,1049,2023,MING,0.022
326,1049,2023,OFFI,0.061
327,1049,2023,OTHR,0.033
328,1049,2023,RETL,0.021


# Get employment data with FIPS codes

In [38]:
tdm_employment_by_county_df = pd.read_csv('intermediate/tdm_employment_by_county.csv')
tdm_employment_by_county_df

,CO_FIPS,YEAR,TDM_SECTOR,EMPCOUNT,CO_NAME
0,1,2023,AGRI,738,BEAVER
1,1,2023,CONS,278,BEAVER
2,1,2023,FOOD,517,BEAVER
3,1,2023,GVED,810,BEAVER
4,1,2023,HLTH,160,BEAVER
...,...,...,...,...,...
13712,57,2065,MING,169,WEBER
13713,57,2065,OFFI,14763,WEBER
13714,57,2065,OTHR,56307,WEBER
13715,57,2065,RETL,20376,WEBER


# Join tables and calculate home-based jobs

In [39]:
tdm_hbj_df = pd.merge(
    tdm_employment_by_county_df,
    wah_melted_df,
    on=['CO_FIPS', 'YEAR', 'TDM_SECTOR'],
    how='inner'
)
tdm_hbj_df

,CO_FIPS,YEAR,TDM_SECTOR,EMPCOUNT,CO_NAME,HBJPCT
0,1,2023,AGRI,738,BEAVER,0.011
1,1,2023,CONS,278,BEAVER,0.051
2,1,2023,FOOD,517,BEAVER,0.012
3,1,2023,GVED,810,BEAVER,0.074
4,1,2023,HLTH,160,BEAVER,0.074
...,...,...,...,...,...,...
314,57,2023,MING,210,WEBER,0.026
315,57,2023,OFFI,11496,WEBER,0.054
316,57,2023,OTHR,39215,WEBER,0.027
317,57,2023,RETL,16106,WEBER,0.023


In [40]:
# do banker's roundign to calculate HBJ counts
tdm_hbj_df['HBJ'] = np.round(
    tdm_hbj_df['EMPCOUNT'] * tdm_hbj_df['HBJPCT']
).astype(int)

tdm_hbj_df['NONHBJ'] = (tdm_hbj_df['EMPCOUNT'] - tdm_hbj_df['HBJ']).astype(int)
tdm_hbj_df = tdm_hbj_df[['CO_FIPS','YEAR','TDM_SECTOR','EMPCOUNT','HBJPCT','HBJ','NONHBJ','CO_NAME']]
tdm_hbj_df

,CO_FIPS,YEAR,TDM_SECTOR,EMPCOUNT,HBJPCT,HBJ,NONHBJ,CO_NAME
0,1,2023,AGRI,738,0.011,8,730,BEAVER
1,1,2023,CONS,278,0.051,14,264,BEAVER
2,1,2023,FOOD,517,0.012,6,511,BEAVER
3,1,2023,GVED,810,0.074,60,750,BEAVER
4,1,2023,HLTH,160,0.074,12,148,BEAVER
...,...,...,...,...,...,...,...,...
314,57,2023,MING,210,0.026,5,205,WEBER
315,57,2023,OFFI,11496,0.054,621,10875,WEBER
316,57,2023,OTHR,39215,0.027,1059,38156,WEBER
317,57,2023,RETL,16106,0.023,370,15736,WEBER


In [41]:
tdm_hbj_df.to_csv('intermediate/tdm_employment_by_county_with_hbj.csv', index=False)

## RUN CHECKS

In [42]:
# 1. Before HBJ: totals from tdm_employment_by_county_df
before_df = (
    tdm_employment_by_county_df
      .groupby(['CO_FIPS', 'YEAR'], as_index=False)
      .agg(EMPCOUNT_BEFORE_HBJ=('EMPCOUNT', 'sum'))
)

# 2. After HBJ: totals from tdm_hbj_df
after_df = (
    tdm_hbj_df
      .groupby(['CO_FIPS', 'YEAR'], as_index=False)
      .agg(
          EMPCOUNT_AFTER_HBJ=('HBJ', 'sum'),
          NONHBJ_AFTER_HBJ=('NONHBJ', 'sum')
      )
      .assign(
          TOTAL_AFTER_HBJ=lambda d: d['EMPCOUNT_AFTER_HBJ'] + d['NONHBJ_AFTER_HBJ']
      )
)

# 3. Combine and compute difference
tdm_hbj_check_df = (
    before_df
      .merge(after_df, on=['CO_FIPS', 'YEAR'], how='outer')
      .assign(
          DIFF=lambda d: d['EMPCOUNT_BEFORE_HBJ'] - d['TOTAL_AFTER_HBJ']
      )
)

# show where differences exist
tdm_hbj_check_df[tdm_hbj_check_df['DIFF'] != 0]


,CO_FIPS,YEAR,EMPCOUNT_BEFORE_HBJ,EMPCOUNT_AFTER_HBJ,NONHBJ_AFTER_HBJ,TOTAL_AFTER_HBJ,DIFF
1,1,2024,4869,NaN,NaN,NaN,NaN
2,1,2025,5031,NaN,NaN,NaN,NaN
3,1,2026,5273,NaN,NaN,NaN,NaN
4,1,2027,5299,NaN,NaN,NaN,NaN
5,1,2028,5325,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
1242,57,2061,219476,NaN,NaN,NaN,NaN
1243,57,2062,220517,NaN,NaN,NaN,NaN
1244,57,2063,221561,NaN,NaN,NaN,NaN
1245,57,2064,222610,NaN,NaN,NaN,NaN


### ONLY 2023 DATA FOR NOW

In [44]:
if tdm_hbj_check_df[(tdm_hbj_check_df['DIFF'] != 0) & (tdm_hbj_check_df['YEAR']==2023)].size == 0:
    print("HBJ calculation check passed: No differences found for 2023 data.")  


HBJ calculation check passed: No differences found for 2023 data.
